# Bots Educacionais para Opções na B3 (Tutorial + Exemplos de Código)Este notebook fornece um roteiro completo **apenas para fins educacionais** sobre como projetar, testar e estruturar **bots de negociação de opções na B3**. O foco é didático: mostrar arquitetura, fluxos de dados, exemplos de estratégias e backtests simplificados. Nada aqui constitui recomendação de investimento ou promessa de resultado; operar derivativos envolve risco elevado, inclusive perda total do capital investido.

## Visão Geral da Arquitetura de um Bot de OpçõesUma arquitetura mínima costuma incluir:1. **Módulo de Dados**: leitura de preços do ativo-objeto e cadeia de opções (option chain) em `pandas.DataFrame`.2. **Módulo de Sinais (Signal Engine)**: transforma a lógica da estratégia em sinais de entrada/saída, aplicando filtros de liquidez, prazo até o vencimento e volatilidade.3. **Módulo de Execução (Execution Layer)**: abstrai envio de ordens. Aqui usamos **paper trading**; para produção seria necessário integrar APIs oficiais de corretoras.4. **Módulo de Gestão de Risco**: limites de capital por trade, perda diária máxima, exposição por ativo e stop após N perdas seguidas.5. **Módulo de Registro e Métricas (Logging & Analytics)**: grava trades, calcula P&L, drawdown, taxa de acerto e relação risco/retorno.Fluxo textual ("diagrama"):```Carregar dados -> Gerar sinais -> Filtrar risco -> Simular execução -> Registrar trade -> Calcular métricas```

In [ ]:
# Skeleton educativo: módulos principais de um bot de opçõesfrom dataclasses import dataclassfrom typing import Dict, List, Optionalimport pandas as pd@dataclassclass RiskConfig:    max_capital_per_trade: float    max_daily_loss: float    max_positions: int = 3    stop_after_losses: int = 3class DataModule:    def load_underlying(self, path: str) -> pd.DataFrame:        """Lê OHLCV do ativo-objeto (ex.: PETR4)."""        return pd.read_csv(path, parse_dates=['date']).set_index('date')    def load_option_chain(self, path: str) -> pd.DataFrame:        """Lê cadeia de opções com colunas como: symbol, type, strike, maturity, bid, ask, volume."""        return pd.read_csv(path, parse_dates=['date', 'maturity'])class SignalEngine:    def long_call_breakout(self, underlying: pd.DataFrame, option_chain: pd.DataFrame, ma_window=21, vol_floor=0.2):        """Exemplo de geração de sinal: compra call direcional quando preço > média móvel e volatilidade acima do piso."""        signals = []        close = underlying['close']        ma = close.rolling(ma_window).mean()        vol = close.pct_change().rolling(21).std() * (252 ** 0.5)        latest_date = close.index[-1]        cond = (close.iloc[-1] > ma.iloc[-1]) and (vol.iloc[-1] >= vol_floor)        if cond:            chain_today = option_chain[option_chain['date'] == latest_date]            liquid_calls = chain_today[(chain_today['type'] == 'CALL') & (chain_today['volume'] > 100)]            if not liquid_calls.empty:                otm = liquid_calls.loc[liquid_calls['strike'] > close.iloc[-1]].sort_values('strike').head(1)                if not otm.empty:                    row = otm.iloc[0]                    signals.append({'action': 'BUY_CALL', 'symbol': row['symbol'], 'price': row['ask']})        return signalsclass ExecutionLayer:    def __init__(self):        self.fills: List[Dict] = []    def send_order(self, symbol: str, qty: int, side: str, price: Optional[float] = None):        """Exemplo didático: registra um fill simulado. Em produção: integrar API da corretora."""        self.fills.append({'symbol': symbol, 'qty': qty, 'side': side, 'price': price})class RiskManager:    def __init__(self, cfg: RiskConfig):        self.cfg = cfg        self.daily_loss = 0.0        self.loss_streak = 0    def allow_trade(self, trade_cost: float, open_positions: int) -> bool:        if trade_cost > self.cfg.max_capital_per_trade:            return False        if self.daily_loss <= -self.cfg.max_daily_loss:            return False        if open_positions >= self.cfg.max_positions:            return False        if self.loss_streak >= self.cfg.stop_after_losses:            return False        return Trueclass TradeLogger:    def __init__(self):        self.trades: List[Dict] = []    def log(self, event: Dict):        self.trades.append(event)    def to_frame(self) -> pd.DataFrame:        return pd.DataFrame(self.trades)

## Três Estratégias Educacionais em Bots de OpçõesCada estratégia abaixo traz a **lógica em linguagem natural**, um **pseudocódigo** e um **exemplo em Python** orientado a backtest. Todas assumem dados preparados em DataFrames e usam apenas simulação.

### 1) Bot de Long Call Direcional**Ideia**: comprar call quando o ativo rompe uma média móvel e a volatilidade anualizada (histórica) está acima de um piso.**Regras principais**- Entrada: preço > média móvel de 21 dias **e** volatilidade ≥ piso definido.- Filtros: opção com volume > 100, strike levemente OTM, maturidade mais próxima acima de 15 dias.- Saída: alvo de ganho em % do prêmio ou stop de perda em %.**Pseudocódigo**```para cada dia:    calcular média móvel e volatilidade    se preço > média e vol >= piso:        escolher call OTM líquida        se filtros de risco ok: comprar 1 lote    monitorar posição:        se prêmio >= alvo -> vender        se prêmio <= -stop -> vender        se faltam < 5 dias para vencimento -> encerrar```

In [ ]:
import numpy as npdef long_call_signals(df_under, df_opts, ma_win=21, vol_floor=0.2, target=0.4, stop=-0.3):    trades = []    close = df_under['close']    ma = close.rolling(ma_win).mean()    vol = close.pct_change().rolling(21).std() * np.sqrt(252)    position = None    entry_price = None    for date in close.index:        px = close.loc[date]        if position:            opt = df_opts.loc[(df_opts['symbol'] == position) & (df_opts['date'] == date)]            if opt.empty:                continue            premium = opt['last'].iloc[0]            ret = (premium - entry_price) / entry_price            if ret >= target or ret <= stop or (opt['days_to_expiry'].iloc[0] <= 5):                trades.append({'date': date, 'symbol': position, 'action': 'EXIT', 'premium': premium, 'ret': ret})                position = None                entry_price = None        else:            cond = (px > ma.loc[date]) and (vol.loc[date] >= vol_floor)            if cond:                todays_chain = df_opts[df_opts['date'] == date]                candidates = todays_chain[(todays_chain['type'] == 'CALL') & (todays_chain['volume'] > 100) & (todays_chain['days_to_expiry'] > 15)]                otm = candidates[candidates['strike'] > px].sort_values('strike').head(1)                if not otm.empty:                    row = otm.iloc[0]                    position = row['symbol']                    entry_price = row['ask']                    trades.append({'date': date, 'symbol': position, 'action': 'ENTRY', 'premium': entry_price, 'basis': 'breakout+vol'})    return pd.DataFrame(trades)

### 2) Bot de Trava de Alta Automatizada (Bull Call Spread)**Ideia**: quando o ativo estiver em tendência de alta moderada, montar uma trava de alta (compra de call ITM e venda de call OTM), limitando perda e ganho.**Regras principais**- Entrada: preço acima de duas médias (curta > longa). Volatilidade não muito alta para evitar prêmios caros.- Montagem: compra call ITM, venda call OTM com mesmo vencimento; calcular risco máximo (débito pago) e payoff máximo (diferença de strikes - débito).- Saída: alvo de ganho parcial ou desmonte se o ativo perder a média longa.**Pseudocódigo**```se MA_curta > MA_longa e vol <= teto:    escolher vencimento > 20 dias    comprar call ITM mais próxima    vender call OTM mais próximaregistrar risco = débito pago; payoff_max = (strike_otm - strike_itm) - débito```

In [ ]:
def bull_call_spread(df_under, df_opts, ma_fast=9, ma_slow=21, vol_cap=0.35):    trades = []    close = df_under['close']    ma_f = close.rolling(ma_fast).mean()    ma_s = close.rolling(ma_slow).mean()    vol = close.pct_change().rolling(21).std() * np.sqrt(252)    for date in close.index:        if not (ma_f.loc[date] > ma_s.loc[date] and vol.loc[date] <= vol_cap):            continue        chain = df_opts[df_opts['date'] == date]        chain = chain[chain['days_to_expiry'] > 20]        calls = chain[chain['type'] == 'CALL']        if calls.empty:            continue        px = close.loc[date]        itm = calls[calls['strike'] <= px].sort_values('strike', ascending=False).head(1)        otm = calls[calls['strike'] > px].sort_values('strike').head(1)        if itm.empty or otm.empty:            continue        buy = itm.iloc[0]        sell = otm.iloc[0]        debit = buy['ask'] - sell['bid']        payoff_max = (sell['strike'] - buy['strike']) - debit        trades.append({            'date': date,            'strategy': 'bull_call_spread',            'buy': buy['symbol'],            'sell': sell['symbol'],            'debit': debit,            'payoff_max': payoff_max,            'risk': debit,        })    return pd.DataFrame(trades)

### 3) Bot de Proteção (Protective Put)**Ideia**: quando a carteira em ações cair além de um limite, o bot compra puts para proteção por um período definido.**Regras principais**- Entrada: drawdown da carteira ≥ limiar (ex.: -5%).- Montagem: comprar put com delta moderado (-0.3 a -0.5) e vencimento > 20 dias.- Saída: desmontar proteção quando carteira recuperar ou faltar menos de 7 dias para o vencimento.**Pseudocódigo**```monitorar valor da carteirase drawdown <= -limite:    escolher put com delta ~ -0.4 e vencimento > 20 dias    comprar quantidade proporcional à exposiçãose carteira recuperar ou maturidade < 7 dias: vender put```

In [ ]:
def protective_put(portfolio_series, df_opts, dd_limit=-0.05):    trades = []    peak = portfolio_series.expanding().max()    drawdown = (portfolio_series - peak) / peak    position = None    for date in portfolio_series.index:        dd = drawdown.loc[date]        if position:            opt = df_opts.loc[(df_opts['symbol'] == position) & (df_opts['date'] == date)]            if opt.empty:                continue            if dd > -0.02 or opt['days_to_expiry'].iloc[0] < 7:                trades.append({'date': date, 'action': 'SELL_PROTECTION', 'symbol': position, 'premium': opt['bid'].iloc[0]})                position = None        else:            if dd <= dd_limit:                todays = df_opts[df_opts['date'] == date]                candidates = todays[(todays['type'] == 'PUT') & (todays['days_to_expiry'] > 20)]                if candidates.empty:                    continue                chosen = candidates.iloc[0]                position = chosen['symbol']                trades.append({'date': date, 'action': 'BUY_PROTECTION', 'symbol': position, 'premium': chosen['ask'].iloc[0], 'drawdown': dd})    return pd.DataFrame(trades)

## Backtesting Simplificado1. **Preparar dados**: DataFrame diário do ativo-objeto e cadeia de opções com `date`, `symbol`, `type`, `strike`, `maturity`, `bid`, `ask`, `last`, `volume`, `days_to_expiry`.2. **Iterar no tempo**: aplicar a função de estratégia dia a dia.3. **Simular entradas e saídas**: registrar prêmios pagos/recebidos e calcular P&L.4. **Calcular métricas**: P&L acumulado, drawdown, número de trades, taxa de acerto, relação risco/retorno.

In [ ]:
from typing import Callabledef run_backtest(strategy_fn: Callable, *dataframes, **kwargs) -> pd.DataFrame:    """Executa uma estratégia que retorna DataFrame de trades simulados."""    trades = strategy_fn(*dataframes, **kwargs)    if trades.empty:        return trades    trades = trades.copy()    if 'ret' not in trades:        if 'debit' in trades:            trades['ret'] = trades['payoff_max'] / trades['debit']        elif 'premium' in trades:            trades['ret'] = trades['premium'].pct_change().fillna(0)        else:            trades['ret'] = 0    trades['cum_pnl'] = trades['ret'].cumsum()    trades['cum_max'] = trades['cum_pnl'].cummax()    trades['drawdown'] = trades['cum_pnl'] - trades['cum_max']    return trades

## Execução Real (Somente Conceitual)Para operar de verdade na B3 é indispensável:- Ter conta em **corretora habilitada** e acesso à **API oficial**.- Configurar **autenticação** (chaves, tokens) e seguir as políticas de risco da corretora.- Respeitar regras da **CVM** e da **B3**; homologar em ambiente de simulação (paper trading) antes de qualquer ordem real.Exemplo de esqueleto apenas ilustrativo:

In [ ]:
class BrokerAPIClient:    def __init__(self, api_key: str, secret_key: str):        # inicialização didática        self.api_key = api_key        self.secret_key = secret_key    def get_quotes(self, symbol: str):        # buscar cotação - exemplo didático        raise NotImplementedError    def send_order(self, symbol: str, qty: int, side: str, order_type: str, price: Optional[float] = None):        # enviar ordem simulada        raise NotImplementedError

## Gestão de Risco, Compliance e Alertas- **Derivativos = alto risco**: possibilidade de perda total do prêmio e, em certas estruturas, prejuízo superior ao capital destinado.- **Liquidez e slippage**: spreads amplos podem distorcer resultados de backtest vs. execução real.- **Disciplina**: bots não substituem gestão humana; defina limites (capital por trade, perda diária, número máximo de posições) e pare após sequência de perdas.- **Aspectos regulatórios**: operar apenas via corretoras autorizadas; leia e siga as regras da CVM e da B3. Este material não é consultoria de investimentos.Sugestão de checklist para cada nova estratégia:1. Rodar backtest com dados extensos e cenários de estresse.2. Validar liquidez mínima e impacto de custos (corretagem, emolumentos, IR).3. Testar em **paper trading** antes de qualquer ordem real.4. Monitorar métricas (P&L, drawdown, taxa de acerto) e revisar parâmetros periodicamente.